In [121]:
# %% Imports and Path Setup
import os
from pathlib import Path
import pandas as pd
import re 
import numpy as np

In [125]:
# 1. Profile Configuration
input_filename = "KUM03_B51-2_ERT2m.csv"
profile_name = "KUM02_2022-08-18_5m"  # Profile name for the new column

merge_all = True

In [123]:
# %% Load, Process, and Save CSV Data
raw_csv_path = input_dir / input_filename

if not raw_csv_path.exists():
    raise FileNotFoundError(
        f"Raw CSV file '{input_filename}' not found in {input_dir}"
    )

# -------------------------------------------------------------
# 1. Load CSV with Auto-Delimiter Detection
# -------------------------------------------------------------
df = pd.read_csv(raw_csv_path, sep=None, engine="python")
df.columns = df.columns.astype(str).str.strip()

# -------------------------------------------------------------
# 2. Map Profile Name & Determine Spacing
# -------------------------------------------------------------
df["profile name"] = profile_name

# Parse default spacing from filename tag like '_W5m' or '_W3m'
match = re.search(r"_W(\d+(?:\.\d+)?)m", profile_name, re.IGNORECASE)
parsed_spacing = float(match.group(1)) if match else 5.0

# Special Spacing Rule for GOL01
if "GOL01" in profile_name.upper():
    elec_spacing = 5.0
    geo_spacing = 2.5
else:
    elec_spacing = parsed_spacing
    geo_spacing = parsed_spacing

df["electrode_spacing"] = elec_spacing
df["geophone_spacing"] = geo_spacing


# -------------------------------------------------------------
# 3. Parse 'Name' Column for Electrodes vs Geophones
# -------------------------------------------------------------
def parse_sensor_name(val):
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])

    val_str = str(val).strip()

    # Geophone (starts with 'G' or 'g')
    if val_str.upper().startswith("G"):
        num_part = re.sub(r"[^\d]", "", val_str)
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([np.nan, sensor_id])  # [electrodes, geophones]

    # Electrode (starts with 'E' or 'e')
    elif val_str.upper().startswith("E"):
        num_part = re.sub(r"[^\d]", "", val_str)
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([sensor_id, np.nan])  # [electrodes, geophones]

    # Pure numbers or fallback -> Electrodes
    else:
        num_part = re.sub(r"[^\d]", "", val_str)
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([sensor_id, np.nan])  # [electrodes, geophones]


name_col = next((c for c in df.columns if c.lower() == "name"), None)

if name_col:
    df[["electrodes", "geophones"]] = df[name_col].apply(parse_sensor_name)
else:
    df["electrodes"] = range(1, len(df) + 1)
    df["geophones"] = np.nan

# Convert to nullable integer type
df["electrodes"] = df["electrodes"].astype("Int64")
df["geophones"] = df["geophones"].astype("Int64")


# -------------------------------------------------------------
# 4. Compute 'x_m' (Profile Distance in Meters)
# -------------------------------------------------------------
def compute_x_pos(row):
    # If electrode ID is present: x = (electrode_id - 1) * electrode_spacing
    if pd.notna(row["electrodes"]):
        return (row["electrodes"] - 1) * row["electrode_spacing"]
    # If geophone ID is present: x = (geophone_id - 1) * geophone_spacing
    elif pd.notna(row["geophones"]):
        return (row["geophones"] - 1) * row["geophone_spacing"]
    else:
        return np.nan


df["x_m"] = df.apply(compute_x_pos, axis=1)

# -------------------------------------------------------------
# 5. Ensure All Target Columns Exist
# -------------------------------------------------------------
target_columns = [
    "profile name",
    "electrodes",
    "geophones",
    "x_m",
    "electrode_spacing",
    "geophone_spacing",
    "Longitude",
    "Latitude",
    "Ellipsoidalheight",
    "Averaging start",
]

# Match existing GPS columns case-insensitively
col_mapping = {c.lower(): c for c in df.columns}
for target in target_columns:
    if target not in df.columns:
        if target.lower() in col_mapping:
            df.rename(
                columns={col_mapping[target.lower()]: target}, inplace=True
            )
        else:
            df[target] = np.nan

# -------------------------------------------------------------
# 6. Filter & Export Cleaned CSV
# -------------------------------------------------------------
df_clean = df[target_columns].copy()

print("\n--- Cleaned Data Preview ---")
print(df_clean.head(10))

output_file = output_dir / f"{profile_name}.csv"
df_clean.to_csv(output_file, index=False)

print(f"\n✓ Saved cleaned file -> {output_file}")


--- Cleaned Data Preview ---
          profile name  electrodes  geophones   x_m  electrode_spacing  \
0  KUM02_2022-08-18_5m           1       <NA>   0.0                5.0   
1  KUM02_2022-08-18_5m           2       <NA>   5.0                5.0   
2  KUM02_2022-08-18_5m           3       <NA>  10.0                5.0   
3  KUM02_2022-08-18_5m           4       <NA>  15.0                5.0   
4  KUM02_2022-08-18_5m           5       <NA>  20.0                5.0   
5  KUM02_2022-08-18_5m           6       <NA>  25.0                5.0   
6  KUM02_2022-08-18_5m           7       <NA>  30.0                5.0   
7  KUM02_2022-08-18_5m           8       <NA>  35.0                5.0   
8  KUM02_2022-08-18_5m           9       <NA>  40.0                5.0   
9  KUM02_2022-08-18_5m          10       <NA>  45.0                5.0   

   geophone_spacing  Longitude   Latitude  Ellipsoidalheight  \
0               5.0  78.067839  41.814600                NaN   
1               5.0  78.067

In [124]:
# %% Load, Process, and Save CSV Data

raw_csv_path = input_dir / input_filename

if not raw_csv_path.exists():
    raise FileNotFoundError(f"Raw CSV file '{input_filename}' not found in {input_dir}")

# -------------------------------------------------------------
# 1. Load CSV with Auto-Delimiter Detection
# -------------------------------------------------------------
# sep=None and engine="python" automatically detects ',' or ';' or '\t'
df = pd.read_csv(raw_csv_path, sep=None, engine="python")

# Clean whitespace from column names (e.g. " Name " -> "Name")
df.columns = df.columns.astype(str).str.strip()

# -------------------------------------------------------------
# 2. Map Profile Name
# -------------------------------------------------------------
df["profile name"] = profile_name

# -------------------------------------------------------------
# 3. Parse 'Name' Column for Electrodes vs Geophones
# -------------------------------------------------------------
def parse_sensor_name(val):
    if pd.isna(val):
        return pd.Series([np.nan, np.nan])
    
    val_str = str(val).strip()
    
    # Check if starts with 'G' or 'g' (Geophone)
    if val_str.upper().startswith("G"):
        num_part = re.sub(r"[^\d]", "", val_str)  # Extract digits only
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([np.nan, sensor_id])  # [electrodes, geophones]
    
    # Check if starts with 'E' or 'e' (Electrode)
    elif val_str.upper().startswith("E"):
        num_part = re.sub(r"[^\d]", "", val_str)
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([sensor_id, np.nan])  # [electrodes, geophones]
    
    # Pure numbers or fallback -> Electrodes
    else:
        num_part = re.sub(r"[^\d]", "", val_str)
        sensor_id = int(num_part) if num_part else np.nan
        return pd.Series([sensor_id, np.nan])  # [electrodes, geophones]

# Locate 'Name' column case-insensitively
name_col = next((c for c in df.columns if c.lower() == "name"), None)

if name_col:
    df[["electrodes", "geophones"]] = df[name_col].apply(parse_sensor_name)
else:
    df["electrodes"] = range(1, len(df) + 1)
    df["geophones"] = np.nan

# -------------------------------------------------------------
# 4. Ensure All Target Columns Exist
# -------------------------------------------------------------
target_columns = [
    "profile name",
    "electrodes",
    "geophones",
    "Longitude",
    "Latitude",
    "Ellipsoidal height",
    "Averaging start",
]

# Match existing columns case-insensitively (e.g., 'longitude' -> 'Longitude')
col_mapping = {c.lower(): c for c in df.columns}
for target in target_columns:
    if target not in df.columns:
        if target.lower() in col_mapping:
            df.rename(columns={col_mapping[target.lower()]: target}, inplace=True)
        else:
            df[target] = np.nan

# -------------------------------------------------------------
# 5. Filter & Reorder strictly to Target Columns
# -------------------------------------------------------------
df_clean = df[target_columns].copy()

# Convert electrodes/geophones to nullable integer type
df_clean["electrodes"] = df_clean["electrodes"].astype("Int64")
df_clean["geophones"] = df_clean["geophones"].astype("Int64")

# Preview clean dataframe
print("\n--- Cleaned Data Preview ---")
print(df_clean.head(10))

# -------------------------------------------------------------
# 6. Export Cleaned CSV
# -------------------------------------------------------------
output_file = output_dir / f"{profile_name}.csv"
df_clean.to_csv(output_file, index=False)

print(f"\n✓ Saved cleaned file -> {output_file}")


--- Cleaned Data Preview ---
          profile name  electrodes  geophones  Longitude   Latitude  \
0  KUM02_2022-08-18_5m           1       <NA>  78.067839  41.814600   
1  KUM02_2022-08-18_5m           2       <NA>  78.067893  41.814570   
2  KUM02_2022-08-18_5m           3       <NA>  78.067947  41.814553   
3  KUM02_2022-08-18_5m           4       <NA>  78.068006  41.814531   
4  KUM02_2022-08-18_5m           5       <NA>  78.068052  41.814503   
5  KUM02_2022-08-18_5m           6       <NA>  78.068079  41.814478   
6  KUM02_2022-08-18_5m           7       <NA>  78.068126  41.814456   
7  KUM02_2022-08-18_5m           8       <NA>  78.068174  41.814434   
8  KUM02_2022-08-18_5m           9       <NA>  78.068217  41.814412   
9  KUM02_2022-08-18_5m          10       <NA>  78.068276  41.814382   

   Ellipsoidal height                  Averaging start  
0            3536.244  2022-08-17 15:21:01.8 UTC+06:00  
1            3536.266  2022-08-17 15:21:35.9 UTC+06:00  
2            3536

In [130]:
if merge_all == True:
    # %% Merge All Cleaned CSVs with Sensor Type Classification
    import pandas as pd
    from pathlib import Path

    csv_dir = (
        repo_root / "data" / "geophysics" / "raw" / "profile_coordinates_topo"
    )
    all_csv_files = list(csv_dir.glob("*.csv"))

    desired_columns = [
        "profile name",
        "electrodes",
        "geophones",
        "Longitude",
        "Latitude",
        "Ellipsoidal height",
    ]

    df_list = []

    for file_path in all_csv_files:
        if file_path.name.startswith("ALL_PROFILES"):
            continue

        df_temp = pd.read_csv(file_path, sep=None, engine="python")
        df_temp.columns = df_temp.columns.astype(str).str.strip()

        col_mapping = {c.lower(): c for c in df_temp.columns}

        for col in desired_columns:
            if col not in df_temp.columns:
                if col.lower() in col_mapping:
                    df_temp.rename(
                        columns={col_mapping[col.lower()]: col}, inplace=True
                    )
                else:
                    df_temp[col] = pd.NA

        df_clean = df_temp[desired_columns].copy()
        df_clean["electrodes"] = df_clean["electrodes"].astype("Int64")
        df_clean["geophones"] = df_clean["geophones"].astype("Int64")

        # --- Add Sensor Type Classification ---
        def classify_sensor(row):
            if pd.notna(row["geophones"]):
                return "Geophone"
            elif pd.notna(row["electrodes"]):
                return "Electrode"
            else:
                return "Unknown"

        df_clean["sensor_type"] = df_clean.apply(classify_sensor, axis=1)

        df_list.append(df_clean)

    master_df = pd.concat(df_list, ignore_index=True)

    master_output_path = csv_dir / "ALL_PROFILES_master.csv"
    master_df.to_csv(master_output_path, index=False)

    print(f"✓ Output saved with sensor_type column -> {master_output_path}")

✓ Output saved with sensor_type column -> C:\Users\mathyst\OneDrive - Université de Fribourg\projects\central-asia-permafrost-geophysics\data\geophysics\raw\profile_coordinates_topo\ALL_PROFILES_master.csv
